# 05_model_registry: Register the Best Model
Register the GBT model in the Unity Catalog Model Registry, document it, run batch predictions, and write the model card (REQ-05).

In [0]:
%run ./00_config

## Step 1: Register the final GBT model

In [0]:
import mlflow
from mlflow import MlflowClient

# Use the Unity Catalog model registry
mlflow.set_registry_uri("databricks-uc")
mlflow.set_experiment(EXPERIMENT_PATH)
client = MlflowClient()

# Find the newest run named "final_gbt"
runs = mlflow.search_runs(
    experiment_names=[EXPERIMENT_PATH],
    filter_string="tags.mlflow.runName = 'final_gbt'",
    order_by=["start_time DESC"],
    output_format="list",
)
run_id = runs[0].info.run_id

# Register only if this run isn't registered yet (safe to re-run)
versions = client.search_model_versions(f"name = '{MODEL_NAME}'")
existing = [v for v in versions if v.run_id == run_id]
mv = existing[0] if existing else mlflow.register_model(f"runs:/{run_id}/model", MODEL_NAME)

print(f"{MODEL_NAME}, version {mv.version}")

Registered the final GBT run in `workspace.loan_default.loan_default_gbt`. Each retraining creates a new version
(the first run created version 1; a full re-run created version 2). Re-running this cell without retraining does not create duplicates.

## Step 2: Document the model and mark version 1 as "champion"

In [0]:
from mlflow import MlflowClient
client = MlflowClient()

description = """Predicts the probability that a Lending Club personal loan will default (Charged Off/Default).
Model: Gradient-Boosted Trees (50 rounds, depth 5) inside a Spark ML Pipeline (cleaning, median imputation, one-hot encoding).
Training data: 547,821 finished loans issued 2012-2015 (Lending Club, Databricks sample). Class weights for imbalance (defaults x3.82).
Inputs (20 raw application-time columns): loan_amnt, term, int_rate, installment, grade, purpose, emp_length, home_ownership,
annual_inc, verification_status, dti, delinq_2yrs, inq_last_6mths, open_acc, pub_rec, revol_bal, revol_util, total_acc,
mort_acc, pub_rec_bankruptcies. No post-loan (leakage) columns.
Test performance (2016 loans): AUC-ROC 0.706, recall 68.3%, precision 33.3% at cut-off 0.5.
Beats a 'reject grades D-G' rule at the same reject rate (recall 50.0% vs 47.8%)."""

client.update_registered_model(MODEL_NAME, description=description)
client.set_registered_model_alias(MODEL_NAME, "champion", mv.version)
print(f"Description added; version {mv.version} is now @champion")

Added a model description (purpose, data, inputs, test performance) and set the alias **@champion** on the latest version.

## Step 3: Load the @champion model by name and run batch predictions on the 2016 test set

In [0]:
import mlflow.spark
from pyspark.sql import functions as F
from pyspark.ml.functions import vector_to_array
from pyspark.ml.evaluation import BinaryClassificationEvaluator

# Load the registered model by its NAME and alias (no run ID needed)
champion = mlflow.spark.load_model(f"models:/{MODEL_NAME}@champion",
                                   dfs_tmpdir=f"/Volumes/{CATALOG}/{SCHEMA}/artifacts/mlflow_tmp")

# Batch scoring: raw test loans in, risk score + decision out
scored = (champion.transform(spark.table(TEST_TABLE))
          .withColumn("p_default", F.round(vector_to_array("probability")[1], 3)))

# Check it matches 04_evaluation (AUC-ROC 0.706)
auc = BinaryClassificationEvaluator(metricName="areaUnderROC").evaluate(scored)
print(f"Scored {scored.count():,} loans | AUC-ROC: {auc:.3f}")
display(scored.select("loan_amnt", "grade", "int_rate", "term", "p_default", "prediction", "label").limit(10))

Loaded the registered model by name (`@champion`) and scored all **98,301** test loans in one batch.
AUC-ROC **0.706**, identical to the evaluation notebook, so the registered model is the same model.
The Pipeline accepts raw application data (e.g. `"  5.32%"`, `" 36 months"`) and handles all cleaning itself.

# 📇 Model Card: `loan_default_gbt` (@champion)

## Intended use
Score **new personal-loan applications** with a probability of default, to support (not replace) credit analysts' approve/reject decisions.
Output: `p_default` (0–1) and a decision at a cut-off chosen by the business.

## Model and data
- **Model:** Gradient-Boosted Trees (50 rounds, depth 5) in a Spark ML Pipeline (cleaning → median imputation → one-hot encoding → GBT).
- **Training data:** 547,821 finished Lending Club loans issued **2012–2015**. Target: Charged Off/Default = 1, Fully Paid = 0.
- **Inputs:** 20 application-time columns (loan amount, term, interest rate, grade, income, debt-to-income, credit history…). No post-loan (leakage) columns.
- **Imbalance:** class weights (defaults ×3.82).

## Performance (test: 2016 loans, never seen in training)
- AUC-ROC **0.706**; at cut-off 0.5: recall 68.3%, precision 33.3% (rejects ~45%; too strict for most lenders).
- At cut-off 0.60: rejects ~29%, catches 50% of defaults, precision 38% (vs 22% random).
- **Beats a "reject grades D–G" rule** at the same reject rate (recall 50.0% vs 47.8%).

## Known limitations
1. **No credit score (FICO)** in the data, a key risk signal, so performance is moderate (AUC ~0.71).
2. **Relies heavily on Lending Club's interest rate** (22% of importance), which already encodes the platform's own risk pricing.
   A lender without that pricing would see weaker results.
3. **Old data (2012–2016):** borrowers, the economy and lending rules have changed since. Deploying in 2026 risks
   **data drift** (applicants look different) and **concept drift** (the link between features and default has changed).
4. **Incomplete recent outcomes:** 2016–2017 loans were only partly finished when the data was saved, so recent years are underrepresented.
5. **Probabilities are not calibrated:** class weights push risk scores upward, so `p_default` is a ranking score, not a true probability.
6. **Fairness:** location (state, zip code) was excluded on purpose, but other inputs may still correlate with protected groups. Not yet audited.
7. **Serving note:** some inputs are integer columns; a real-time API should accept missing values as floats.

## Before production
- Retrain and validate on **recent** data; set a **retraining schedule**.
- **Choose the cut-off with the credit-risk team** using real costs (loss per default vs profit per good loan).
- **Calibrate** probabilities if they will be used for pricing or provisioning.
- Run a **fairness audit** and check legal requirements (e.g. explaining rejections to applicants).
- **Monitor** input drift and live default rates; compare new versions as challengers against @champion.
- Add a credit score (FICO) or bureau data if available.